<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 60 Interface

The notebook checks that every button, menu, field and setting of 3D-Baukasten 1.3 is in the page of
version 2.0, at the place where 1.3 has it, and that every icon is an SVG. The sources are the forms
of 1.3 in the repository: the menu bar of `3D_bau.frm`, the tool bar `F3.frm` with its thirteen
buttons and their tool tips, the body bar `F2.frm`, the register form `F4.frm` with six tabs in two
rows, the bar `F5.frm` and the window F6, which the thesis shows. Then it checks the start of the
page, the drawer of a narrow screen, the keys, what the browser keeps, the icons, the roles and the
contrast, the window F6, the texts of the status field, the dialog Hilfe and the submenu Beispiele,
the latched Start, the hatch of the frame FARBE and the swatch of the tab Oberfläche. The page opens
in headless Chromium with WebGPU on SwiftShader, in the modern look at 1440 by 900, and section 9
opens a second page at 768 by 1024.

A check names the ID of the row of `Documentation/quirks.md` it proves, and a code cell carries the
tag of the column Test, such as `boot` for `60:boot`. A check that `expect` makes waits for the last
cell when it fails, so that a defect found early still lets the sections after it run. The parameter
`write_docs` copies the three renders of section 18 to `Documentation/images/`.

In [ ]:
width = 1440
height = 900
look = 'modern'
write_docs = False

In [ ]:
import sys, json, math, re
sys.path.insert(0, '..')
from harness import *

In [ ]:
site = Site()
page = Browser(site.url, width=width, height=height, look=look)
page.open()
DEFERRED = []


def expect(condition, message, *context):
    """Checks as check() does, and keeps a failure for the last cell instead of stopping here."""
    try:
        check(condition, message, *context)
    except Failure as failure:
        DEFERRED.append(str(failure))
        print(f"FAIL {message}")


def fresh(look=None):
    """Empties the scene through Datei › Neu and answers Ja, as a user does."""
    page.eval("() => { bausteine.act('neu'); }")
    page.settle()
    if page.eval("() => document.querySelector('#msgbox').open"):
        page.page.click('#msgbox .c-default')
        page.settle()
    page.key('Escape')
    page.key('Escape')


check(page.errors() == [], 'the page starts without an error', page.errors())


def at(view, x, y, z):
    """Answers the pixel of the page that shows the point (x, y, z) in the view `view`."""
    return page.eval(
        "([v, x, y, z]) => { const r = document.querySelector(`.view[data-view='${v}']`).getBoundingClientRect();"
        " const b = bausteine.app.bridge; const p = b.project(v, x, y, z); if (!p) return null;"
        " const w = b.cameras ? b.cameras[v * 18 + 16] : 0; const inset = w > 0 ? Math.max(0, (r.width - w) / 2) : 0;"
        " return [r.left + inset + p[0], r.top + inset + p[1]]; }", [view, x, y, z])


def hover_world(view, x, y, z):
    """Moves the pointer over the point (x, y, z) of the view `view`."""
    p = at(view, x, y, z)
    page.page.mouse.move(p[0], p[1])
    page.settle()
    return p

## 1. The menu bar of Form1

Version 1.3 has the menus Datei, Bearbeiten, Objekte and Ansicht, in this order, and Datei holds
Neu, Öffnen, Speichern, Speichern unter, Schließen and Beenden with the shortcuts Strg+N, Strg+O and
Strg+S. Version 2.0 keeps them in their order, fills the empty menus and adds Hilfe at the end.

In [ ]:
titles = page.texts('#menubar .menu-title')
check(titles == ['Datei', 'Bearbeiten', 'Objekte', 'Ansicht', 'Hilfe'], 'Q-5 the four menus of 1.3 in their order, Hilfe behind them', titles)
datei = page.texts('[data-menu="datei"] > .menu-popup > .menu-entry', 'data-entry')
original = [e for e in datei if e in ('neu', 'oeffnen', 'speichern', 'speichern-unter', 'schliessen', 'beenden')]
check(original == ['neu', 'oeffnen', 'speichern', 'speichern-unter', 'schliessen', 'beenden'], 'Datei keeps the six entries of 1.3 in their order', datei)
check(datei[:5] == original[:5] and datei[-1] == 'beenden', 'the new entries stand between Schließen and Beenden, so no entry of 1.3 moves', datei)
check(page.texts('[data-entry="neu"] .menu-shortcut') == ['Strg+N'] and page.texts('[data-entry="oeffnen"] .menu-shortcut') == ['Strg+O'] and page.texts('[data-entry="speichern"] .menu-shortcut') == ['Strg+S'], 'the shortcuts of 1.3')
bearbeiten = page.texts('[data-menu="bearbeiten"] .menu-entry', 'data-entry')
check(bearbeiten == ['rueckgaengig', 'wiederholen', 'verdoppeln', 'loeschen', 'nichts-waehlen', 'pov-kopieren'], 'Q-5 Bearbeiten holds Rückgängig, Wiederholen, Verdoppeln, Löschen, Auswahl aufheben and POV-Text kopieren', bearbeiten)
objekte = page.texts('[data-menu="objekte"] .menu-entry', 'data-body')
check(objekte == ['Kugel', 'Kegel', 'Zylinder', 'Würfel', 'Quader', 'Pyramide', 'Prisma', 'Disc', 'Ebene', 'CSG-Vereinigung', 'CSG-Differenz', 'CSG-Durchschnitt', 'CSG-Summe', 'CSG-Ende', '#Neu', 'Neu_Ende'], 'Q-8 Objekte offers what the body bar and CB1 offer, as the thesis says the menus do', objekte)
ansicht = page.texts('[data-menu="ansicht"] .menu-entry', 'data-entry')
check(ansicht == ['zoom2d-in', 'zoom2d-out', 'zoom3d-in', 'zoom3d-out', 'zoomed-d3', 'fit', 'raytrace', 'look-modern', 'look-classic', 'dark', 'inverted', 'bildschirm', 'bild-speichern', 'fenster-anordnen', 'fenster-zeigen'], 'Q-8 Ansicht holds the zooms, the 3D view over four fields, Alles zeigen, Raytracing, the looks, the inverted views, Bildschirmeinstellungen, Bild speichern and the windows', ansicht)
check(page.texts('[data-menu="hilfe"] .menu-entry', 'data-entry') == ['hilfe', 'info'], 'Q-5 the menu Hilfe at the right end holds Hilfe and Über')

## 2. The tool bar F3

`F3.frm` has thirteen buttons in one column. The first nine carry the tool tips of 1.3 letter by
letter, the last four are blank in 1.3 and host undo, redo, the look and the help in 2.0.

In [ ]:
tips = page.texts('#tb1 .tb-button', 'data-tip')
check(len(tips) == 13, 'thirteen buttons', tips)
check(tips[:9] == ['Position', 'Körper einfügen', 'Bearbeiten', 'Farbe ändern', 'Raytrace', '3D-Zoom', 'Ausgabe', 'Beenden', 'Dia-Show'], 'T-p97c the tool tips of 1.3', tips)
check(tips[9:] == ['Rückgängig (Strg+Z)', 'Wiederholen (Strg+Y)', 'Darstellung: Modern oder Klassisch', 'Hilfe (?)'], 'Q-77 the places 10 to 13 hold Rückgängig, Wiederholen, the switch of the look and Hilfe', tips[9:])
svg = page.eval("() => [...document.querySelectorAll('#tb1 .tb-button')].map((b) => b.querySelector(':scope > svg') !== null)")
check(all(svg), 'every tool carries an SVG icon', svg)
boxes = [page.rect(f'#tb1 [data-tool="{n}"]') for n in range(1, 14)]
check(all(abs(a[0] - boxes[0][0]) < 1 for a in boxes) and all(b[1] > a[1] for a, b in zip(boxes, boxes[1:])), 'the tools stand in one column, top to bottom')

## 3. The body bar F2

`F2.frm` has nine buttons whose keys are the tool tips: Kugel, Kegel, Zylinder, Würfel, Quader,
Pyramide, Prisma, Disc and Ebene. Disc and Ebene have no picture in 1.3, 2.0 gives them one.

In [ ]:
bodies = page.texts('#tb2 .tb-button', 'data-tip')
check(bodies == ['Kugel', 'Kegel', 'Zylinder', 'Würfel', 'Quader', 'Pyramide', 'Prisma', 'Disc', 'Ebene'], 'T-p92a the nine bodies of the source in their order', bodies)
svg = page.eval("() => [...document.querySelectorAll('#tb2 .tb-button')].map((b) => { const s = b.querySelector(':scope > svg'); return s ? s.querySelectorAll('path, rect, circle, ellipse, polygon').length : 0; })")
check(all(n > 0 for n in svg), 'Q-68 every body carries a drawn SVG icon, Disc and Ebene included', svg)
images = page.eval("() => document.querySelectorAll('img:not(#f9-picture)').length")
check(images == 0, 'Q-73 no bitmap of 1.3 is shipped, the icons are SVG', images)

## 4. The register form F4

`F4.frm` holds an SSTab with six tabs in two rows: Körper, Oberfläche and Ausgabe in front, Bilder,
Kamera and Einstellung behind. Each tab keeps its controls and the wording of its captions, and a
few controls of 2.0 explain themselves with a tool tip.

In [ ]:
check(page.texts('#sstab1 .tab-row.back .tab') == ['Bilder', 'Kamera', 'Einstellung'], 'F4-Q12 the back row, its last tab captioned Einstellung')
check(page.texts('#sstab1 .tab-row.front .tab') == ['Körper', 'Oberfläche', 'Ausgabe'], 'the front row')
controls = {
    'koerper': ['#frame8', '#cb2', '#frame1', '#cb1', '#l1-1', '#t1-1', '#t1-2', '#t1-3', '#l1-4', '#l1-6', '#t1-4', '#t1-5', '#t1-6', '#l1-7', '#t1-7', '#t1-8', '#t1-9', '#btn-einf', '#btn-change', '#btn-clear'],
    'ausgabe': ['#frame5', '#text1', '#tfname', '#bpov-comp', '#bpov-save'],
    'bilder': ['#file1', '#tdatei', '#bbild-anz', '#bbild-beenden', '#cmdia'],
    'kamera': ['#label1-kamera', '#frame6', '#hs-4', '#hst-4', '#hs-1', '#hst-1', '#hs-2', '#hst-2', '#hs-3', '#hst-3', '#start', '#command2', '#btn1-1', '#btn2-1'],
    'einstellung': ['#tpasswd', '#lpasswd', '#frame10', '#tbau-home', '#tpov-home', '#tbsp-home', '#tbsp-def', '#bscreen', '#bdatei-change', '#bdatei-def'],
}
page.entry('Pyramide')  # a mask with every caption of the rows of position and size
for tab, ids in controls.items():
    page.page.click(f'#sstab1 [data-tab="{tab}"]')
    page.settle()
    hidden = [i for i in ids if not page.visible(i)]
    check(hidden == [], f'the tab {tab} shows its controls of 1.3', hidden)
page.page.click('#sstab1 [data-tab="koerper"]')
check(page.texts('#frame8 legend') == ['OBJEKTE:'] and page.texts('#frame1 legend') == ['GEOMETRIEDATEN:'], 'F4-Q24 the captions of the frames of 1.3')
check(page.texts('#btn-einf, #btn-change, #btn-clear') == ['Einfügen', 'Ändern', 'Löschen'], 'F4-Q24 the three buttons of 1.3')
check(page.texts('#label1-kamera') == ['noch in der Testphase!!'] and page.texts('#label1-kamera', 'title') == ['So steht es in Version 1.3'], 'F4-Q4 the note of the tab Kamera stays, with the tool tip So steht es in Version 1.3')
check(page.texts('#bpov-comp, #bpov-save') == ['Übersetzen', 'Ausgeben'], 'the buttons of the tab Ausgabe')
tips = page.eval("() => ['#cmdia', '#pov-command', '#btn1-1', '#btn2-1'].map((s) => document.querySelector(s).getAttribute('title') || '')")
check(all(tips) and tips[0] == 'Bild hinzufügen', 'F4-Q20 the button … of tab Bilder, the command line of tab Ausgabe and the buttons Einfügen and Löschen of tab Kamera explain themselves', tips)
page.entry('Prisma')
check(page.texts('#l1-4') == ['Kantenlängen x y'], 'F4-Q22 the mask of Prisma reads Kantenlängen x y', page.texts('#l1-4'))
page.entry('Quader')

## 5. The bar F5 and the window F6

`F5.frm` holds, left to right, the fields x, y and z, the 3D zoom with `<` and `>`, the 2D zoom with
its field and `<` and `>`, and the status. The fields of a normal view show the two coordinates of
the view, the modern bar keeps the third greyed with its last value, and a right or a middle drag
pans while the wheel zooms. F6 holds the list TEXTUR and the frame FARBE with its button Farbe.

In [ ]:
order = ['#tx', '#ty', '#tz', '#d3-zoomin', '#d3-zoomout', '#tzoom', '#d2-zoomin', '#d2-zoomout', '#tstatus']
lefts = [page.rect(s)[0] for s in order]
check(lefts == sorted(lefts), 'T-p88b the controls of F5 stand in the order of 1.3, under the views', lefts)
check(page.rect('#f5')[1] >= page.rect('#views')[1] + page.rect('#views')[3] - 2, 'T-p88b the bar of 1.3 lies under the views')
check(page.texts('#d3-zoomin, #d3-zoomout, #d2-zoomin, #d2-zoomout') == ['<', '>', '<', '>'], 'Q-89 the captions < and > of 1.3')
zoom_tips = page.texts('#d3-zoomin, #d3-zoomout, #d2-zoomin, #d2-zoomout', 'title')
check(zoom_tips == ['3D hineinzoomen', '3D herauszoomen', '2D hineinzoomen', '2D herauszoomen'], 'Q-89 tool tips say which button zooms in and which out', zoom_tips)
heights = [round(page.rect(s)[3]) for s in ('#tx', '#ty', '#tz')]
check(len(set(heights)) == 1, 'Q-92 the three fields share one height', heights)
check(page.eval("() => ['#tx', '#ty', '#tz', '#tzoom'].every((s) => document.querySelector(s).readOnly)"), 'Q-93 the fields of F5 are read only')
fields = "() => ['#tx', '#ty', '#tz'].map((s) => [document.querySelector(s).value, document.querySelector(s).classList.contains('dim')])"
hover_world(TOP, 3, 4, 0)
grund = page.eval(fields)
check(grund[0] == ['3', False] and grund[1] == ['4', False] and grund[2][1] is True, 'T-p90b the Grundriss fills x and y, and the modern bar keeps z greyed', grund)
hover_world(FRONT, 0, 2, 5)
auf = page.eval(fields)
check(auf == [['3', True], ['2', False], ['5', False]], 'T-p90b the Aufriss fills y and z and keeps the last x greyed', auf)
r = page.rect('.view[data-view="2"]')
cx, cy = r[0] + r[2] / 2, r[1] + r[3] / 2
for button in ('right', 'middle'):
    before = page.eval("() => bausteine.app.bridge.project(2, 0, 0, 0)")
    page.page.mouse.move(cx, cy); page.page.mouse.down(button=button)
    page.page.mouse.move(cx + 40, cy + 30, steps=5); page.page.mouse.up(button=button); page.settle()
    after = page.eval("() => bausteine.app.bridge.project(2, 0, 0, 0)")
    check(abs(after[0] - before[0] - 40) < 2 and abs(after[1] - before[1] - 30) < 2, f'Q-51 a {button} drag pans the Grundriss', before, after)
z = page.eval("() => bausteine.app.zoomf")
page.page.mouse.move(cx, cy); page.page.mouse.wheel(0, -120); page.settle()
check(abs(page.eval("() => bausteine.app.zoomf") - z * 1.2) < 1e-9, 'Q-51 the wheel zooms', z, page.eval("() => bausteine.app.zoomf"))
page.page.mouse.wheel(0, 120); page.settle()
hover_world(FRONT, 0, 2, 5)
oblique = page.rect('.view[data-view="3"]')
page.page.mouse.move(oblique[0] + oblique[2] / 2, oblique[1] + oblique[3] * 0.6); page.settle()
ground = page.eval(fields)
check(page.texts('#label1-f5') == ['3D'], 'the pointer stands over the Schrägriss')
expect(ground != auf and ground[2][0] in ('0', ''), 'Q-51 over the Schrägriss the modern bar shows the point under the pointer on the ground plane', auf, ground)
check(page.texts('#frame11 legend') == ['TEXTUR'] and page.texts('#bcolor') == ['Farbe'], 'F6 holds TEXTUR and Farbe')
options = page.texts('#cb3 option')
check(options[0] == 'Keine' and len(options) == 44 and options[1] == 'Chrome_Metal', 'M-63 Keine and the 43 textures of 1.3', options[:3])

## 6. The places of the windows

`Form_Set` puts the tool bar at the top left, the body bar under it, Form1 in the middle, F4 at the
right edge with F6 under it, and F5 under Form1. `ViewControl` puts the Kreuzriss at the top left,
the Aufriss at the top right, the Schrägriss at the bottom left and the Grundriss at the bottom
right.

In [ ]:
r = {k: page.rect(k) for k in ['#f3', '#f2', '#form1', '#f4', '#f6', '#f5', '#views']}
check(r['#f3'][0] < r['#form1'][0] and r['#f3'][1] <= r['#f2'][1] and r['#f3'][2] < r['#f3'][3], 'Q-87 F3 is a vertical bar at the top left, F2 under it', r['#f3'], r['#f2'])
check(abs(r['#f2'][0] - r['#f3'][0]) < 2, 'F2 in the column of F3')
check(r['#f4'][0] > r['#form1'][0] + r['#form1'][2] - 2, 'F4 at the right edge', r['#f4'])
check(r['#f6'][1] >= r['#f4'][1] + r['#f4'][3] - 2 and abs(r['#f6'][0] - r['#f4'][0]) < 2, 'F6 under F4', r['#f6'])
check(r['#f5'][1] >= r['#views'][1] + r['#views'][3] - 2 and r['#f5'][0] >= r['#form1'][0] - 1, 'F5 under the views of Form1', r['#f5'])
v = {n: page.rect(f'.view[data-view="{n}"]') for n in range(4)}
check(v[1][0] < v[0][0] and v[1][1] < v[3][1] and v[3][0] < v[2][0] and v[0][1] < v[2][1], 'T-p89c Kreuzriss top left, Aufriss top right, Schrägriss bottom left, Grundriss bottom right', v)

## 7. Pictures

The worker draws the four views with WebGPU after each change, and every frame holds all four views
whole. The page shows in the light and the dark theme of the modern look.

In [ ]:
def drawn():
    """Answers the number of colours the read back picture holds in each of the four views."""
    return {n: len(page.sample_box(*page.rect(f'.view[data-view="{n}"]'))) for n in range(4)}

revision = page.revision()
page.page.click('#cb2 [data-line="1"]'); page.settle()
page.page.fill('#t1-1', '2'); page.page.click('#btn-change')
check(page.wait_revision(revision + 1) > revision, 'Q-34 the worker draws a new frame after the change', revision, page.revision())
colours = drawn()
check(all(n > 2 for n in colours.values()), 'Q-35 the frame after the change holds all four views drawn whole', colours)
page.key('Control+z'); page.settle()
path, stats = page.screenshot('60_modern_light')
check(stats['lit'] > 0, 'Q-34 the worker draws the views with WebGPU', stats)
page.eval("() => bausteine.act('dark')")
page.settle()
path, stats = page.screenshot('60_modern_dark')
check(page.eval("() => document.documentElement.dataset.theme") == 'dark', 'the dark theme')
page.eval("() => bausteine.act('dark')")
check(page.errors() == [], 'no error on the way', page.errors())

## 8. The start

While WebAssembly and WebGPU start, a card in the layout of the splash of 1.3 shows and closes at
the end of the start, and Hilfe › Über shows it again until a click or a key. The page then holds
the start state of 1.3: the 2D zoom 1.2 and the 3D zoom 1.0, the tab Körper in front with `Quader`
in CB1, the fields `0 0 0`, `1 1 1` and `0 0 0`, Ausgeben disabled and `0 Raster` as the first line
of the list. Where WebGPU is missing, the page shows a German text and a button Neu laden.

In [ ]:
page.page.goto(f"{site.url}/?debug&offscreen", wait_until='commit')
page.page.wait_for_function("() => { const s = document.querySelector('#splash'); return s && !s.hidden && document.documentElement.dataset.ready !== '1'; }", timeout=20000)
check(page.eval("() => document.querySelector('#splash-state').textContent").startswith('WebAssembly und WebGPU starten'), 'Q-103 the card of the splash shows while WebAssembly and WebGPU start')
page.page.wait_for_function("() => window.bausteine && bausteine.app.scene.id && document.documentElement.dataset.ready === '1'", timeout=60000)
page.settle()
check(page.eval("() => document.querySelector('#splash').hidden"), 'Q-103 and closes itself at the end of the start')
check(page.eval("() => document.documentElement.dataset.ready") == '1' and page.revision() >= 0, 'the root carries data-ready and the revision of the last frame', page.revision())
check(page.eval("() => bausteine.app.bridge.protocol") == 1, 'the worker answers ready with the protocol 1')
page.eval("() => bausteine.act('info')"); page.settle()
check(page.texts('.splash-title, .splash-version, .splash-origin, .splash-copyright') == ['3D-Bausteine', 'Version 2.0', 'nach 3D-Baukasten 1.3 von Herbert Kleber (2002)', 'Florian Kleber 2026'], 'Q-103 Hilfe › Über shows the card with its factual lines')
page.page.click('.splash-version'); page.settle()
check(page.eval("() => document.querySelector('#splash').hidden"), 'Q-103 a click closes the card')
start = page.eval("() => [bausteine.app.zoomf, bausteine.app.d3zoomf, document.querySelector('#tzoom').value]")
check(start == [12, 10, '120 %'], 'Q-29 the 2D views start at 1.2 and the 3D view at 1.0', start)
check(page.eval("() => document.querySelector('#f4').dataset.tab") == 'koerper' and page.eval("() => document.querySelector('#cb1-edit').value") == 'Quader', 'the tab Körper in front, Quader in CB1')
check(page.fields()[:9] == ['0', '0', '0', '1', '1', '1', '0', '0', '0'], 'the fields 0 0 0, 1 1 1 and 0 0 0', page.fields())
check(page.eval("() => document.querySelector('#bpov-save').disabled") is True, 'Ausgeben is disabled')
check(page.lines()[0] == '0 Raster', 'F4-Q30 the list shows 0 Raster from the start', page.lines()[:2])
check(page.eval("() => document.documentElement.dataset.mode") == 'position', 'the root mirrors the mode Position')
fallback = page.eval("() => { const f = document.querySelector('#fallback'); return [f.hidden, f.querySelector('h2').textContent, document.querySelector('#fallback-reload').textContent.trim()]; }")
check(fallback == [True, 'WebGPU ist hier nicht verfügbar', 'Neu laden'], 'R-54 without WebGPU the page shows the German text and offers Neu laden', fallback)

## 9. The drawer of a narrow screen

Below 1024 px the modern look turns F4 and F6 into a drawer over the views. The drawer stays closed
at the start, opens whenever 1.3 brings F4 to the front, such as a click on the body bar, closes on
Esc and on a click outside, and the toggle at the right end of the caption strip opens and closes
it. A second page opens at 768 by 1024.

In [ ]:
small = Browser(site.url, width=768, height=1024, look='modern')
small.open()
state = "() => { const d = document.querySelector('.dock-right'); const c = getComputedStyle(d); const t = document.querySelector('#toggle-right').getBoundingClientRect(); const hit = document.elementFromPoint(t.left + t.width / 2, t.top + t.height / 2); return { open: d.hasAttribute('data-open') && c.visibility === 'visible', toggle: hit !== null && hit.closest('#toggle-right') !== null }; }"
check(small.eval("() => getComputedStyle(document.querySelector('.dock-right')).position") == 'fixed', 'the right dock is a drawer below 1024 px')
check(small.visible('#toggle-right'), 'the toggle shows at the right end of the caption strip')
views = {n: small.rect(f'.view[data-view="{n}"]') for n in range(4)}
check(views[1][1] == views[0][1] and views[3][1] == views[2][1] and views[1][0] == views[3][0], 'the views keep the arrangement of two by two', views)
expect(small.eval(state)['open'] is False, 'the drawer is closed at the start', small.eval(state))
small.page.click('#tb2 [data-body="Kugel"]'); small.settle()
check(small.eval(state)['open'] is True, 'a click on the body bar brings F4 to the front in the drawer', small.eval(state))
small.page.keyboard.press('Escape'); small.settle()
expect(small.eval(state)['open'] is False, 'Esc closes the drawer', small.eval(state))
small.eval("() => document.querySelector('.dock-right').setAttribute('data-open', '')"); small.settle()
small.page.mouse.click(200, 400); small.settle()
expect(small.eval(state)['open'] is False, 'a click outside closes the drawer', small.eval(state))
small.eval("() => document.querySelector('.dock-right').setAttribute('data-open', '')"); small.settle()
expect(small.eval(state)['toggle'] is True, 'the toggle stays reachable while the drawer is open', small.eval(state))
check(small.errors() == [], 'no error on the narrow page', small.errors())
small.close()

## 10. The keys

The keys of 1.3 and of 2.0: Strg+N, Strg+O and Strg+S of the menu Datei, Strg+Umschalt+S, Strg+Z,
Strg+Y, Strg+Umschalt+Z and Strg+D, Entf, Esc in two steps, Enter while a body is armed, 1 to 4 and
0 for one view or all, F, R, `+` and `−` for the 2D ladder, Alt with `+` and `−` for the 3D ladder,
the arrows and Bild↑ and Bild↓ to move the selected body, `?` and F1 for the help, and Alt with the
hot letters of the menus. A key does nothing while a text field has the focus, Esc and Enter
excepted. Raytrace stays on the tool bar, in Ansicht and on the key R.

In [ ]:
fresh()
blur = "() => document.activeElement && document.activeElement.blur()"
page.page.click('#tb2 [data-body="Würfel"]'); page.settle()
page.eval(blur)
n = page.eval("() => bausteine.app.scene.items.length")
page.key('Enter'); page.settle()
check(page.eval("() => bausteine.app.scene.items.length") == n + 1 and page.status().startswith('Eingefügt: '), 'Enter inserts the armed body', page.status())
page.key('Escape')
check(page.eval("() => bausteine.app.mode") == 'position' and page.eval("() => bausteine.app.line") >= 0, 'the first Esc disarms and keeps the selection')
page.key('Escape')
check(page.eval("() => bausteine.app.line") == -1, 'the second Esc clears the selection')
page.key('Control+z'); page.settle()
check(page.lines() == ['0 Raster'], 'Strg+Z takes the insert back', page.lines())
page.key('Control+y'); page.settle()
check(page.lines() == ['0 Raster', '1 Würfel'], 'Strg+Y does it again', page.lines())
page.key('Control+z'); page.settle(); page.key('Control+Shift+z'); page.settle()
check(page.lines() == ['0 Raster', '1 Würfel'], 'Strg+Umschalt+Z does it again as well', page.lines())
page.page.click('#cb2 [data-line="1"]'); page.settle(); page.eval(blur)
page.key('Control+d'); page.settle()
check(page.lines() == ['0 Raster', '1 Würfel', '2 Würfel'], 'Strg+D doubles the selected body behind it', page.lines())
page.key('Delete'); page.settle()
check(page.lines() == ['0 Raster', '1 Würfel'], 'Entf removes the selected body', page.lines())
page.page.click('#cb2 [data-line="1"]'); page.settle()
page.page.locator('#views').hover(); page.eval(blur)
position = lambda: page.eval("() => bausteine.app.scene.items[0].position")
p0 = position()
page.key('ArrowRight'); page.settle(); page.key('PageUp'); page.settle()
p1 = position()
check(p1 != p0 and p1['z'] == p0['z'] + 1, 'the arrows and Bild↑ move the selected body by one unit', p0, p1)
page.key('PageDown'); page.settle()
for key, view in (('1', 1), ('2', 0), ('3', 2), ('4', 3), ('0', None)):
    page.key(key)
    check(page.eval("() => bausteine.app.maximized") == view, f'the key {key} shows one view or all', page.eval("() => bausteine.app.maximized"))
z = page.eval("() => bausteine.app.zoomf")
page.key('+'); check(abs(page.eval("() => bausteine.app.zoomf") - z * 1.2) < 1e-9, 'the plus key steps up the 2D ladder')
page.key('-'); check(abs(page.eval("() => bausteine.app.zoomf") - z) < 1e-9, 'the minus key steps down the 2D ladder')
d = page.eval("() => bausteine.app.d3zoomf")
page.key('Alt++'); check(abs(page.eval("() => bausteine.app.d3zoomf") - d * 1.2) < 1e-9, 'Alt with the plus key steps up the 3D ladder')
page.key('Alt+-'); check(abs(page.eval("() => bausteine.app.d3zoomf") - d) < 1e-9, 'Alt with the minus key steps down the 3D ladder')
page.page.click('#cb2 [data-line="1"]'); page.settle(); page.eval(blur)
page.eval("() => { bausteine.app.zoomf = 30; }")
page.key('f'); page.settle()
check(page.eval("() => bausteine.app.zoomf") != 30, 'F shows all bodies')
check(page.texts('#tb1 [data-tool="5"]', 'data-tip') == ['Raytrace'] and page.texts('[data-entry="raytrace"] .menu-shortcut') == ['R'], 'F4-Q76 Raytrace stays on the tool bar and in Ansicht with the key R')
page.key('r'); page.settle()
check(page.eval("() => bausteine.app.raytrace") is True, 'F4-Q76 the key R starts the ray tracer')
page.key('r'); page.settle()
check(page.eval("() => bausteine.app.raytrace") is False, 'and stops it again')
for key in ('?', 'F1'):
    page.key(key)
    check(page.eval("() => document.querySelector('#help-dialog').open"), f'{key} opens the help')
    page.key('Escape')
check(not page.eval("() => document.querySelector('#help-dialog').open"), 'Esc closes a dialog')
page.key('Control+o'); page.settle()
check(page.eval("() => document.querySelector('#open-dialog').open"), 'Strg+O opens the dialog Öffnen')
page.key('Escape')
page.key('Control+n'); page.settle()
check(page.texts('#msgbox-title') == ['FRAGE'], 'Strg+N asks the question of 1.3')
page.page.click('#msgbox-buttons button:nth-child(2)'); page.settle()
with page.page.expect_download() as download:
    page.page.keyboard.press('Control+s')
check(download.value.suggested_filename.endswith('.pov'), 'Strg+S saves the POV file as 1.3 did', download.value.suggested_filename)
with page.page.expect_download() as download:
    page.page.keyboard.press('Control+Shift+s')
check(download.value.suggested_filename.endswith('.bausteine.json'), 'Strg+Umschalt+S saves the scene file', download.value.suggested_filename)
opened = []
for letter in 'dboah':
    page.key(f'Alt+{letter}')
    opened += page.eval("() => [...document.querySelectorAll('#menubar .menu-item > .menu-popup')].filter((p) => !p.hidden).map((p) => p.parentElement.dataset.menu)")
    page.key('Escape')
check(opened == ['datei', 'bearbeiten', 'objekte', 'ansicht', 'hilfe'], 'Alt with D, B, O, A and H opens the menus', opened)
page.page.focus('#t1-1')
page.key('2')
check(page.eval("() => bausteine.app.maximized") is None, 'a key typed into a text field does not act')
page.page.click('#tb2 [data-body="Kugel"]'); page.settle()
page.page.focus('#t1-1')
page.key('Escape')
check(page.eval("() => bausteine.app.mode") == 'position', 'Esc acts inside a text field and disarms the body')

## 11. What the browser keeps

The browser keeps every scene across reloads, under the address `#/szene/<key>`. A reload starts an
empty service in the worker, and the page rebuilds the scene from its copy. `#s=<key>` opens a scene
of the browser, `#d=<data>` opens a copy of a shared scene under a key of its own. Speichern unter
writes the scene file `.bausteine.json`, a declaration travels with it, and Öffnen reads it, an
example or a scene of the browser. Schließen asks first when changes are unsaved, and the settings
stay across a reload.

In [ ]:
import base64, zlib
fresh()
page.entry('#Neu'); page.page.click('#btn-einf'); page.settle()
page.page.click('#tb2 [data-body="Kegel"]'); page.page.click('#btn-einf'); page.settle()
page.entry('Neu_Ende'); page.page.click('#btn-einf'); page.settle()
page.entry('Werkstück1'); page.page.click('#btn-einf'); page.settle()
lines, key = page.lines(), page.eval("() => bausteine.app.sceneKey")
check(page.eval("() => location.hash") == f'#/szene/{key}', 'R-52 the address names the scene by the key of its copy', page.eval("() => location.hash"))
service_id = page.eval("() => bausteine.app.scene.id")
page.page.reload()
page.page.wait_for_function("() => window.bausteine && bausteine.app.scene.id && document.documentElement.dataset.ready === '1'", timeout=60000)
page.settle()
check(page.lines() == lines and page.eval("() => bausteine.app.sceneKey") == key, 'Q-6 the browser keeps the scene across a reload', page.lines())
check(page.eval("() => bausteine.app.scene.items.map((i) => i.kind)") == ['DECLARE', 'CONE', 'INSTANCE'], 'R-52 the page rebuilds the scene in the empty service from its copy', service_id, page.eval("() => bausteine.app.scene.id"))
page.page.goto(f"{site.url}/?debug&offscreen#s={key}")
page.page.wait_for_function("() => window.bausteine && bausteine.app.scene.id && document.documentElement.dataset.ready === '1'", timeout=60000)
page.settle()
check(page.eval("() => bausteine.app.sceneKey") == key and page.eval("() => location.hash") == f'#/szene/{key}', '#s= opens the scene of the browser and the address becomes #/szene/<key>')
with page.page.expect_download() as download:
    page.eval("() => { bausteine.act('speichern-unter'); }")
saved = json.loads(open(download.value.path(), encoding='utf-8').read())
check(download.value.suggested_filename.endswith('.bausteine.json') and page.status().endswith('.bausteine.json gespeichert'), 'Q-6 Speichern unter writes the scene file', download.value.suggested_filename, page.status())
check([i['kind'] for i in saved['items']] == ['DECLARE', 'CONE', 'INSTANCE'], 'T-p78b a declaration travels with the scene file', [i['kind'] for i in saved['items']])
page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
page.eval("() => { bausteine.act('schliessen'); }"); page.settle()
check(page.eval("() => document.querySelector('#msgbox').open") and page.texts('#msgbox-title') == ['FRAGE'], 'Q-6 Schließen asks first when changes are unsaved')
page.page.click('#msgbox-buttons button:nth-child(2)'); page.settle()
check(page.lines() == lines + [f'{len(lines)} Kugel'], 'Nein keeps the scene', page.lines())
page.eval("() => { bausteine.act('oeffnen'); }"); page.settle()
offer = page.eval("() => [[...document.querySelectorAll('#open-examples [role=option]')].map((o) => o.textContent), document.querySelectorAll('#open-scenes [role=option]').length]")
check('Haus mit Tür und Fenster' in offer[0] and offer[1] >= 1, 'Q-6 Öffnen offers the examples and the scenes the browser keeps', offer)
page.page.set_input_files('#open-file', download.value.path())
page.until("(want) => JSON.stringify(bausteine.app.lines.map((l) => l.text)) === JSON.stringify(want)", lines)
check(page.lines() == lines, 'Q-6 Öffnen reads the scene file', page.lines())
names = page.eval("() => [...document.querySelectorAll('#cb1 .c-combo-list [data-entry]')].map((e) => e.dataset.entry)")
check(names[-1] == 'Werkstück1', 'T-p78b the declared body of the file stands in CB1', names[-3:])
items = [{'name': 'Quader', 'kind': 'CUBOID', 'parent': None, 'position': {'x': 1, 'y': 2, 'z': 0}, 'size': {'x': 2, 'y': 2, 'z': 2}, 'rotation': {'x': 0, 'y': 0, 'z': 0}, 'color': '#ff0000', 'texture': None, 'innerRadius': 0, 'source': None}]
deflate = zlib.compressobj(9, zlib.DEFLATED, -15)
packed_scene = deflate.compress(json.dumps({'name': 'Geteilt', 'items': items}).encode()) + deflate.flush()
page.page.goto(f"{site.url}/?debug&offscreen#d={base64.urlsafe_b64encode(packed_scene).decode().rstrip('=')}")
page.page.wait_for_function("() => window.bausteine && bausteine.app.scene.id && document.documentElement.dataset.ready === '1'", timeout=60000)
page.settle()
check(page.lines() == ['0 Raster', '1 Quader'] and page.eval("() => bausteine.app.scene.name") == 'Geteilt' and page.eval("() => bausteine.app.sceneKey") != key, '#d= opens a copy of the shared scene under a key of its own', page.lines(), page.eval("() => location.hash"))
page.eval("() => { bausteine.act('bildschirm'); }"); page.settle()
page.page.select_option('#screen-dialog select[name="snap"]', '0.5')
page.page.click('#screen-dialog [value="ok"], #screen-dialog button.c-default'); page.settle()
page.page.reload()
page.page.wait_for_function("() => window.bausteine && bausteine.app.scene.id && document.documentElement.dataset.ready === '1'", timeout=60000)
page.settle()
check(page.eval("() => bausteine.app.settings.snap") == 0.5 and json.loads(page.eval("() => localStorage.getItem('bausteine.display')"))['snap'] == 0.5, 'F4-Q84 the browser stores the settings', page.eval("() => bausteine.app.settings"))
page.eval("() => { bausteine.act('bildschirm'); }"); page.settle()
page.page.select_option('#screen-dialog select[name="snap"]', '1')
page.page.click('#screen-dialog [value="ok"], #screen-dialog button.c-default'); page.settle()

## 12. The icons of the modern look

Every element that names an icon holds an SVG drawing, every SVG of the page is hidden from
assistive technology and draws its lines in the colour of the text, and the favicon and the caption
icon show one motif. A contact sheet of every icon goes to `out/60_icons.png`.

In [ ]:
empty = page.eval("() => [...document.querySelectorAll('[data-icon]')].filter((e) => !e.querySelector('svg')).map((e) => e.dataset.icon)")
check(empty == [], 'every [data-icon] is filled with an SVG', empty)
loud = page.eval("() => [...document.querySelectorAll('svg')].filter((s) => s.getAttribute('aria-hidden') !== 'true').map((s) => s.parentElement.id || s.parentElement.className)")
check(loud == [], 'every SVG is aria-hidden', loud)
plain = page.eval("() => [...document.querySelectorAll('svg')].filter((s) => !s.outerHTML.includes('currentColor')).map((s) => s.parentElement.dataset.icon || s.parentElement.dataset.tool || s.parentElement.dataset.body || s.parentElement.id)")
check(plain == [], 'every SVG draws its lines in currentColor', plain)
favicon = page.eval("() => [document.querySelector('#favicon').getAttribute('href'), document.querySelector('#favicon').type]")
check(favicon == ['icons/favicon.svg', 'image/svg+xml'], 'Q-66 the favicon is an SVG', favicon)
caption = page.eval("() => document.querySelector('#form1 .win-icon svg').outerHTML")
fills = lambda text: set(re.findall(r'fill="(#[0-9a-fA-F]{6})"', text))
favicon_svg = site.get('/icons/favicon.svg').text
expect(fills(caption) & fills(favicon_svg), 'Q-66 the favicon and the caption icon show the same motif', sorted(fills(caption)), sorted(fills(favicon_svg)))
page.eval('''() => {
  const sheet = document.createElement('div');
  sheet.id = 'icon-sheet';
  sheet.style.cssText = 'position:fixed;inset:0;z-index:99999;background:#fff;color:#1c2024;display:flex;flex-wrap:wrap;align-content:flex-start;gap:6px;padding:12px;overflow:hidden';
  const seen = new Set();
  for (const s of document.querySelectorAll('svg')) {
    const html = s.outerHTML;
    if (seen.has(html)) continue;
    seen.add(html);
    const cell = document.createElement('div');
    cell.style.cssText = 'width:40px;height:40px;display:grid;place-items:center;border:1px solid #e1e3df;border-radius:5px';
    cell.innerHTML = html;
    const c = cell.firstElementChild;
    c.setAttribute('width', '28'); c.setAttribute('height', '28');
    sheet.append(cell);
  }
  document.body.append(sheet);
}''')
page.page.screenshot(path=str(OUT / '60_icons.png'))
count = page.eval("() => document.querySelectorAll('#icon-sheet > div').length")
page.eval("() => document.querySelector('#icon-sheet').remove()")
check(count >= 30, 'the contact sheet holds every icon once', count)

## 13. Roles and contrast

The menu bar, the menus, the two tool bars, the register with its tabs and pages, the lists, the
tool tip, the status and the dialogs carry their roles, every button has a name, and the texts of
the modern tokens keep a contrast of at least 4.5 to their backgrounds in the light and in the dark
theme.

In [ ]:
roles = page.eval('''() => ({
  menubar: document.querySelectorAll('#menubar[role=menubar]').length,
  menus: document.querySelectorAll('#menubar .menu-popup[role=menu]').length,
  toolbars: document.querySelectorAll('#tb1[role=toolbar], #tb2[role=toolbar]').length,
  tablists: document.querySelectorAll('#sstab1 [role=tablist]').length,
  tabs: document.querySelectorAll('#sstab1 [role=tab]').length,
  panels: document.querySelectorAll('#sstab1 [role=tabpanel]').length,
  listboxes: document.querySelectorAll('#cb2[role=listbox], #file1[role=listbox], #cb1 [role=listbox]').length,
  tooltip: document.querySelectorAll('#tooltip[role=tooltip]').length,
  status: document.querySelectorAll('#toast[role=status], #splash[role=status]').length,
  live: document.querySelector('#tstatus').getAttribute('aria-live'),
  dialogs: document.querySelectorAll('dialog').length,
})''')
check(roles == {'menubar': 1, 'menus': 6, 'toolbars': 2, 'tablists': 2, 'tabs': 6, 'panels': 6, 'listboxes': 3, 'tooltip': 1, 'status': 2, 'live': 'polite', 'dialogs': roles['dialogs']} and roles['dialogs'] >= 5, 'the controls carry their roles', roles)
nameless = page.eval("() => [...document.querySelectorAll('button')].filter((b) => !(b.textContent.trim() || b.getAttribute('aria-label') || b.getAttribute('title'))).map((b) => b.id || b.className)")
check(nameless == [], 'every button has a name', nameless)


def luminance(colour):
    """Answers the relative luminance of the colour #rrggbb as WCAG 2 defines it."""
    channels = [int(colour[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    linear = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in channels]
    return 0.2126 * linear[0] + 0.7152 * linear[1] + 0.0722 * linear[2]


def contrast(a, b):
    """Answers the contrast ratio of the colours a and b."""
    la, lb = sorted((luminance(a), luminance(b)), reverse=True)
    return (la + 0.05) / (lb + 0.05)


PAIRS = [('--text', '--bg'), ('--text', '--surface'), ('--text', '--surface-2'), ('--text', '--surface-geo'), ('--text-2', '--surface'), ('--accent-text', '--accent'), ('--lcd-x', '--lcd-bg'), ('--lcd-y', '--lcd-bg'), ('--lcd-z', '--lcd-bg')]
for theme in ('light', 'dark'):
    if page.eval("() => document.documentElement.dataset.theme") != theme:
        page.eval("() => bausteine.act('dark')"); page.settle()
    low = [(a, b, round(contrast(page.css('html', a), page.css('html', b)), 2)) for a, b in PAIRS]
    low = [p for p in low if p[2] < 4.5]
    check(low == [], f'the tokens of the {theme} theme keep a contrast of 4.5', low)
if page.eval("() => document.documentElement.dataset.theme") != 'light':
    page.eval("() => bausteine.act('dark')"); page.settle()

## 14. The window F6

F6 holds the frame TEXTUR with the combo CB3 above the frame FARBE with the button Farbe, the order
of the record in the EXE. CB3 carries the only tool tip of 1.3 outside the two bars,
`Textur auswählen`, which the tool tips of the page show, and the frame FARBE takes the colour of
the selected body.

In [ ]:
fresh()
check(page.rect('#frame11')[1] < page.rect('#frame-farbe')[1], 'R-38 TEXTUR stands above FARBE')
check(page.eval("() => [...document.querySelectorAll('#f6 [id]')].map((e) => e.id)") == ['frame11', 'cb3', 'frame-farbe', 'bcolor'], 'R-38 F6 holds the frame TEXTUR with CB3 and the frame FARBE with Farbe')
page.page.hover('#cb3'); page.until("() => !document.querySelector('#tooltip').hidden")
tip = page.eval("() => [document.querySelector('#tooltip').hidden, document.querySelector('#tooltip').textContent]")
check(tip == [False, 'Textur auswählen'], 'R-38 the tool tip of CB3 shows through the tool tips of the page', tip)
page.page.mouse.move(5, 5); page.settle()
grey = page.css('#frame-farbe', 'background-color')
page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
colour = page.eval("() => bausteine.app.scene.items[0].color").lower()
check(page.css('#frame-farbe', 'background-color') == colour and grey != colour, 'R-38 the frame FARBE takes the colour of the selected body', grey, colour)

## 15. The texts of the status field

The status field asks for the marks of 1.3 with the spelling of 1.3, and phase 2 asks with texts of
its own for the size along x, y and z. After phase 2 the modern status says what to press. The
status names an unknown body and the Ebene that this version cannot insert, a known name matches
whatever its case, and Beenden asks first.

In [ ]:
fresh()
page.page.click('#tb2 [data-body="Quader"]'); page.settle()
check(page.status() == 'Bezugspunkt in einem Riss markieren', 'the status asks for the reference point', page.status())
page.click_world(TOP, 2, 3, 0)
check(page.status() == 'z-Koordinate markieren', 'the Grundriss asks for z', page.status())
page.click_world(FRONT, 0, 3, 1)
check(page.status() == 'Grösse angeben oder markieren', 'then for the size, spelled as 1.3 spells it', page.status())
page.click_world(TOP, 6, 7, 0)
check(page.status() == 'Höhe markieren', 'Q-42 after the size in the Grundriss phase 2 asks for the height', page.status())
page.click_world(FRONT, 0, 7, 3)
check(page.status() == 'Einfügen drücken (Enter)', 'Q-42 after phase 2 the modern status says what to press', page.status())
page.key('Escape'); page.page.click('#tb2 [data-body="Quader"]'); page.settle()
page.click_world(SIDE, 2, 0, 1); page.click_world(TOP, 2, 3, 0)
page.click_world(FRONT, 0, 5, 4)
check(page.status() == 'Tiefe markieren', 'Q-42 after a size in the Aufriss phase 2 asks for the depth along x', page.status())
page.key('Escape'); page.page.click('#tb2 [data-body="Quader"]'); page.settle()
page.click_world(TOP, 2, 3, 0); page.click_world(FRONT, 0, 3, 1)
page.click_world(SIDE, 5, 0, 4)
check(page.status() == 'Breite markieren', 'Q-42 after a size in the Kreuzriss phase 2 asks for the width along y', page.status())
page.key('Escape'); page.key('Escape')
page.entry('Klotz')
check(page.status() == 'Unbekannter Körper: Klotz', 'F4-Q41 the status names the unknown body', page.status())
page.entry('kugel')
check(page.eval("() => document.querySelector('#cb1-edit').value") == 'Kugel' and page.texts('#l1-4') == ['Radius'], 'F4-Q41 a known name matches whatever its case', page.texts('#l1-4'))
page.page.click('#tb2 [data-body="Ebene"]'); page.settle()
check(page.status() == 'Ebene ist in dieser Version nicht einfügbar' and page.eval("() => document.querySelector('#cb1-edit').value") == 'Ebene', 'Q-69 the button Ebene puts Ebene into the combo and the status says it is not insertable', page.status())
n = page.eval("() => bausteine.app.scene.items.length")
page.page.click('#btn-einf'); page.settle()
check(page.eval("() => bausteine.app.scene.items.length") == n, 'Q-69 Ebene inserts nothing')
page.key('Escape')
page.eval("() => { bausteine.act('beenden'); }"); page.settle()
check(page.texts('#msgbox-title') == ['FRAGE'] and page.texts('#msgbox-text') == ['Programm beenden?'], 'Q-82 Beenden asks first', page.texts('#msgbox-text'))
page.page.click('#msgbox-buttons button:nth-child(2)'); page.settle()
check(page.visible('#form1') and not page.eval("() => document.querySelector('#msgbox').open"), 'Nein keeps the program')

## 16. The dialog Hilfe and the submenu Beispiele

The page POV-Referenz of the dialog Hilfe sets what the thesis printed in German quotation marks.
The dialog keeps its height while its three pages change, the chosen tab stands in the accent, only
the page scrolls, and the corrected samples keep their lines in a monospace font and scroll
sideways. In the menu Datei the arrow down stops on Beispiele with the submenu closed, the arrow
right opens it with the focus inside, the arrow left closes it with the focus back on Beispiele, and
the pointer still opens it.

In [ ]:
page.eval("() => { bausteine.act('hilfe'); }"); page.settle()
#: Answers the box of the dialog Hilfe, the style of each tab, and for the dialog, its client and the page shown whether each scrolls.
HELP = '''() => {
  const d = document.querySelector('#help-dialog');
  const box = d.getBoundingClientRect();
  const tabs = [...d.querySelectorAll('.help-tab')].map((t) => ({ chosen: t.getAttribute('aria-selected') === 'true', height: t.getBoundingClientRect().height, background: getComputedStyle(t).backgroundColor }));
  const scrolls = (e) => ({ overflow: getComputedStyle(e).overflowY, over: e.scrollHeight > e.clientHeight + 1 });
  return { box: [box.left, box.top, box.width, box.height], tabs, page: scrolls(d.querySelector('.help-page:not([hidden])')), dialog: scrolls(d), client: scrolls(d.querySelector('.help-client')) };
}'''
CALM = "() => document.querySelector('#help-dialog').getAnimations({ subtree: true }).length === 0"
pages = {}
for name in ('bedienung', 'tasten', 'pov'):
    page.page.click(f'#help-dialog [data-help-tab="{name}"]'); page.until(CALM)
    pages[name] = page.eval(HELP)
quotes = page.eval("() => { const q = document.querySelector('#help-dialog q.thesis'); return [getComputedStyle(q, '::before').content, getComputedStyle(q, '::after').content, getComputedStyle(q).quotes]; }")
check(quotes == ['open-quote', 'close-quote', '"„" "“" "‚" "‘"'], 'the page POV-Referenz sets what the thesis printed in German quotation marks', quotes)
check(len({tuple(p['box']) for p in pages.values()}) == 1, 'the dialog Hilfe keeps its place and its height while the three pages change', {n: p['box'] for n, p in pages.items()})
soft = page.css('html', '--accent-soft')
chosen = {n: [hex_colors(t['background']) for t in p['tabs'] if t['chosen']] for n, p in pages.items()}
others = {n: [hex_colors(t['background']) for t in p['tabs'] if not t['chosen']] for n, p in pages.items()}
check(all(c == [soft] and soft not in others[n] for n, c in chosen.items()), 'the chosen tab of the dialog Hilfe stands in the soft accent and the others do not', soft, chosen, others)
check(all(p['page'] == {'overflow': 'auto', 'over': p['page']['over']} and not p['dialog']['over'] and not p['client']['over'] for p in pages.values()) and pages['pov']['page']['over'], 'only the page of the dialog Hilfe scrolls, the dialog and its client keep their size', pages)
code = page.eval("() => { const s = getComputedStyle(document.querySelector('#help-dialog pre.help-code')); return [s.overflowX, s.whiteSpace, s.fontFamily]; }")
check(code[:2] == ['auto', 'pre'] and code[2].endswith('monospace'), 'the samples keep their lines in a monospace font and scroll sideways', code)
page.key('Escape')
check(not page.eval("() => document.querySelector('#help-dialog').open"), 'Esc closes the dialog Hilfe')
SUBMENU = "() => { const e = document.querySelector('#menubar [data-entry=beispiele]'); const m = document.querySelector('#menu-examples'); return { shown: getComputedStyle(m).display !== 'none', expanded: e.getAttribute('aria-expanded'), focus: document.activeElement === e ? 'entry' : m.contains(document.activeElement) ? 'inside' : 'elsewhere' }; }"
page.eval("() => document.activeElement && document.activeElement.blur()")
page.key('Alt+d')
for _ in range(12):
    if page.eval(SUBMENU)['focus'] == 'entry':
        break
    page.key('ArrowDown')
down = page.eval(SUBMENU)
page.key('ArrowRight')
right = page.eval(SUBMENU)
page.key('ArrowLeft')
left = page.eval(SUBMENU)
page.key('Escape'); page.key('Escape')
check(down == {'shown': False, 'expanded': 'false', 'focus': 'entry'}, 'the arrow down stops on Beispiele and leaves its submenu closed', down)
check(right == {'shown': True, 'expanded': 'true', 'focus': 'inside'}, 'the arrow right opens the submenu Beispiele with the focus inside', right)
check(left == {'shown': False, 'expanded': 'false', 'focus': 'entry'}, 'the arrow left closes it with the focus back on Beispiele', left)
page.page.click('#menubar .menu-item[data-menu="datei"] > .menu-title'); page.page.hover('#menubar [data-entry="beispiele"]'); page.settle()
hover = page.eval(SUBMENU)
page.key('Escape'); page.key('Escape')
check(hover['shown'] is True, 'the pointer still opens the submenu Beispiele', hover)

## 17. Start, the hatch of FARBE and the swatch of the tab Oberfläche

While the turn of tab Kamera runs, Start stays latched in the accent. A body with a texture hatches
the colour of the frame FARBE, which keeps the colour of the body, and Keine takes the texture away
again. The swatch of the tab Oberfläche paints the material that the service answers for the texture
of the selected body, in its family and its colours: Jade a green marble, Gold_Metal a gold metal,
Green_Glass a green glass, Cherry_Wood a red brown wood and Candy_Cane white stripes.

In [ ]:
fresh()
page.page.click('#sstab1 [data-tab="kamera"]'); page.settle()
page.page.fill('#hst-3', '90'); page.page.press('#hst-3', 'Enter'); page.settle()
idle = [page.css('#start', 'background-color'), page.css('#start', 'border-top-color')]
page.page.click('#start')
page.until("() => { const s = document.querySelector('#start'); return s.getAttribute('aria-pressed') === 'true' && s.getAnimations().length === 0; }")
latched = [page.css('#start', 'background-color'), page.css('#start', 'border-top-color'), page.css('#start', 'color')]
page.page.click('#command2'); page.settle()
accent = [page.css('html', '--accent-soft'), page.css('html', '--accent'), page.css('html', '--accent')]
check(latched == accent and latched[:2] != idle, 'Start stays latched in the accent while the turn runs', idle, latched, accent)
check(page.texts('#start', 'aria-pressed') == ['false'], 'Stopp releases Start')
page.page.fill('#hst-3', '0'); page.page.press('#hst-3', 'Enter'); page.settle()
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
page.page.click('#tb2 [data-body="Kugel"]'); page.page.click('#btn-einf'); page.settle()
page.page.click('#cb2 [data-line="1"]'); page.settle()
colour = page.eval("() => bausteine.app.scene.items[0].color").lower()
page.page.select_option('#cb3', 'Jade'); page.settle()
FRAME = "() => { const f = document.querySelector('#frame-farbe'); return [f.hasAttribute('data-textured'), getComputedStyle(f).backgroundImage.split('(')[0]]; }"
hatch = page.eval(FRAME)
check(page.eval("() => bausteine.app.scene.items[0].texture") == 'Jade' and hatch == [True, 'repeating-linear-gradient'], 'a body with a texture hatches the frame FARBE', hatch)
check(page.css('#frame-farbe', 'background-color') == colour, 'the hatched frame FARBE keeps the colour of the body', colour, page.css('#frame-farbe', 'background-color'))
page.page.select_option('#cb3', '')
page.settle()
expect(page.eval("() => bausteine.app.scene.items[0].texture") is None and page.eval(FRAME)[0] is False, 'M-63 Keine in CB3 takes the texture of the selected body away', page.eval("() => bausteine.app.scene.items[0].texture"), page.eval(FRAME))
fresh()
page.menu('datei/beispiele/Kugeln mit Texturen')
if page.eval("() => document.querySelector('#msgbox').open"):
    page.page.click('#msgbox .c-default'); page.settle()
page.until("() => bausteine.app.scene.items.length === 16")
MATERIALS = {'Jade': ('marble', (46, 140, 82)), 'Gold_Metal': ('metal', (242, 191, 64)), 'Green_Glass': ('glass', (140, 230, 153)), 'Cherry_Wood': ('wood', (158, 71, 46)), 'Candy_Cane': ('stripes', (242, 242, 242))}
painted = {}
for name, (surface, rgb) in MATERIALS.items():
    line = page.eval("(n) => bausteine.app.lines.findIndex((l) => !l.end && bausteine.app.scene.items.find((i) => i.id === l.item)?.texture === n)", name)
    page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
    page.page.click(f'#cb2 [data-line="{line}"]'); page.settle()
    page.page.click('#sstab1 [data-tab="oberflaeche"]'); page.settle()
    swatch = page.eval("() => { const s = document.querySelector('#surface-swatch'); return [s.dataset.surface ?? null, s.style.background]; }")
    painted[name] = [swatch[0], 'rgb({}, {}, {})'.format(*rgb) in swatch[1]]
check(page.visible('#surface-swatch'), 'F4-Q1 the modern tab Oberfläche shows the swatch of the material of the selected item')
check(painted == {name: [surface, True] for name, (surface, _) in MATERIALS.items()}, 'F4-Q1 the swatch paints the family and the colours of the material that the service answers for the texture', painted)
page.page.click('#sstab1 [data-tab="koerper"]'); page.settle()
fresh()

## 18. Pictures and the end

The last section writes the three renders of 2.0 that `README.md` shows: the example Haus in the
four views, its ray traced Schrägriss, and the same scene in the classic look at 1024 by 768. With
`write_docs` they are copied to `Documentation/images/`. The page logged no error on the way.

In [ ]:
import shutil
IMAGES = ROOT / 'Documentation' / 'images'


def readme(look_name, width, height):
    """Opens the example Haus in a page of the look `look_name` and writes the renders of README.md."""
    shot = Browser(site.url, width=width, height=height, look=look_name)

    def aside():
        """Moves the pointer to the bottom right corner, where no tool tip and no view waits, so that a slow machine draws the same picture."""
        shot.page.mouse.move(width - 2, height - 2)
        shot.settle()

    try:
        shot.open()
        shot.menu('datei/beispiele/Haus mit Tür und Fenster'); shot.settle()
        shot.page.click('#cb2 [data-line="5"]'); aside()
        names = {}
        path, stats = shot.screenshot('readme-modern' if look_name == 'modern' else 'readme-klassisch')
        names['zugeordnete-normalrisse.png' if look_name == 'modern' else 'klassisch.png'] = (path, stats)
        if look_name == 'modern':
            shot.page.click('#cb2 [data-line="0"]'); shot.settle()
            shot.page.click('#tb1 [data-tool="5"]'); aside()
            names['raytracing.png'] = shot.screenshot('readme-raytracing')
        return names, shot.errors()
    finally:
        shot.close()


renders = {}
for look_name, w, h in (('modern', 1440, 900), ('classic', 1024, 768)):
    names, errors = readme(look_name, w, h)
    check(errors == [], f'the renders of the {look_name} look without an error', errors)
    renders.update(names)
check(all(stats['lit'] > 0 for _, stats in renders.values()), 'R-45 the three renders of README.md show the drawn views', {k: v[1] for k, v in renders.items()})
if write_docs:
    for name, (path, _) in renders.items():
        shutil.copyfile(path, IMAGES / name)
    check(all((IMAGES / name).read_bytes() == path.read_bytes() for name, (path, _) in renders.items()), 'the renders are copied to Documentation/images/')
path, stats = page.screenshot('60_modern_end')
check(page.errors() == [], 'no error on the way', page.errors())

In [ ]:
page.close()
site.stop()
check(DEFERRED == [], 'every check that waited for the end holds', *DEFERRED)